# Lab 5: Evaluation -- Debugging Post-Training Failures

In this lab, you will investigate two models that have been fine-tuned on subtly corrupted data. Your goal is to:

1. **Evaluate** both models and observe unexpected performance drops
2. **Investigate** the training data to find suspicious samples
3. **Build verifiers** that automatically detect corrupted training examples
4. **Filter and retrain** to recover model performance

This mirrors a real-world scenario: you receive a fine-tuned model that behaves unexpectedly, and you need to diagnose what went wrong.

**Two models, two failure modes:**
- **Code model**: Fine-tuned on coding problems where some training solutions contain subtle bugs
- **QA model**: Fine-tuned on trivia questions where some training answers are plausible but incorrect

---

## Section 0: Setup and Load

This section loads all the pre-trained artifacts you will need. Nothing to implement here -- just run each cell and verify the outputs look correct.

In [ ]:
# =============================================================================
# Configuration -- Update these paths if your artifacts are in a different location
# =============================================================================

from pathlib import Path

ARTIFACTS_DIR = Path("./lab_artifacts")
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Derived paths
DATA_DIR = ARTIFACTS_DIR / "data"
CODE_TRAIN_PATH = DATA_DIR / "code_train.json"
CODE_EVAL_PATH = DATA_DIR / "code_eval.json"
CODE_POISON_INDICES_PATH = DATA_DIR / "code_poison_indices.json"
QA_TRAIN_PATH = DATA_DIR / "qa_train.json"
QA_EVAL_PATH = DATA_DIR / "qa_eval.json"
QA_POISON_INDICES_PATH = DATA_DIR / "qa_poison_indices.json"

# HuggingFace Dataset paths
CODE_TRAIN_DS_DIR = DATA_DIR / "code_train_dataset"
CODE_EVAL_DS_DIR = DATA_DIR / "code_eval_dataset"
QA_TRAIN_DS_DIR = DATA_DIR / "qa_train_dataset"
QA_EVAL_DS_DIR = DATA_DIR / "qa_eval_dataset"

MANIFEST_PATH = ARTIFACTS_DIR / "manifest.pt"
SEED = 42

print("Configuration loaded.")
print(f"Artifacts directory: {ARTIFACTS_DIR.resolve()}")

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import torch
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import tinker
from tinker import types
from transformers import AutoTokenizer
from datasets import load_from_disk
import textwrap
import re
import subprocess
import tempfile
import os
import warnings

warnings.filterwarnings("ignore")

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Visualization defaults
plt.style.use('seaborn-v0_8-whitegrid')
COLOR_BASE = '#4C72B0'       # blue -- base model
COLOR_POISONED = '#DD8452'   # orange -- poisoned model
COLOR_RETRAINED = '#55A868'  # green -- retrained model

print("All imports loaded successfully.")

In [ ]:
# =============================================================================
# Load Manifest
# =============================================================================

manifest = torch.load(MANIFEST_PATH, map_location="cpu", weights_only=False)

print("=" * 60)
print("Lab Manifest")
print("=" * 60)
print(f"  Base model:        {manifest.get('base_models', {}).get('code', MODEL_NAME)}")
print(f"  Seed:              {manifest['seed']}")
print()
if 'lora_config' in manifest:
    print(f"  LoRA rank:         {manifest['lora_config']['rank']}")
    print(f"  LoRA alpha:        {manifest['lora_config']['alpha']}")
    print(f"  Target modules:    {manifest['lora_config']['target_modules']}")
    print()
print("Dataset statistics:")
for key, value in manifest['dataset_stats'].items():
    print(f"  {key:20s}: {value}")
print()
if 'tinker_models' in manifest:
    print("Tinker model names:")
    for key, value in manifest['tinker_models'].items():
        print(f"  {key:20s}: {value}")
print("=" * 60)

In [ ]:
# =============================================================================
# Load Datasets (JSON)
# =============================================================================

with open(CODE_TRAIN_PATH, "r") as f:
    code_train = json.load(f)

with open(CODE_EVAL_PATH, "r") as f:
    code_eval = json.load(f)

with open(QA_TRAIN_PATH, "r") as f:
    qa_train = json.load(f)

with open(QA_EVAL_PATH, "r") as f:
    qa_eval = json.load(f)

# NOTE: Poison indices are loaded here for later validation of your verifier.
# In a real scenario you would NOT have these labels.
with open(CODE_POISON_INDICES_PATH, "r") as f:
    code_poison_indices = set(json.load(f))

with open(QA_POISON_INDICES_PATH, "r") as f:
    qa_poison_indices = set(json.load(f))

print(f"Code training samples:   {len(code_train)}")
print(f"Code eval samples:       {len(code_eval)}")
print(f"Code poisoned indices:   {len(code_poison_indices)}")
print()
print(f"QA training samples:     {len(qa_train)}")
print(f"QA eval samples:         {len(qa_eval)}")
print(f"QA poisoned indices:     {len(qa_poison_indices)}")

In [ ]:
# =============================================================================
# Load Datasets (HuggingFace format)
# =============================================================================

code_train_ds = load_from_disk(str(CODE_TRAIN_DS_DIR))
code_eval_ds = load_from_disk(str(CODE_EVAL_DS_DIR))
qa_train_ds = load_from_disk(str(QA_TRAIN_DS_DIR))
qa_eval_ds = load_from_disk(str(QA_EVAL_DS_DIR))

print("HuggingFace Datasets loaded:")
print(f"  code_train_ds: {code_train_ds.num_rows} rows, columns: {code_train_ds.column_names}")
print(f"  code_eval_ds:  {code_eval_ds.num_rows} rows, columns: {code_eval_ds.column_names}")
print(f"  qa_train_ds:   {qa_train_ds.num_rows} rows, columns: {qa_train_ds.column_names}")
print(f"  qa_eval_ds:    {qa_eval_ds.num_rows} rows, columns: {qa_eval_ds.column_names}")

In [ ]:
# =============================================================================
# Helper: Model Loading via Tinker
# =============================================================================

def get_tinker_service():
    """Get or create a Tinker ServiceClient."""
    return tinker.ServiceClient()


def load_base_model(model_name):
    """Create a Tinker SamplingClient for a base model (no LoRA).

    Args:
        model_name: Model identifier (e.g., 'Qwen/Qwen3-4B-Instruct-2507')

    Returns:
        tuple: (sampling_client, tokenizer)
    """
    print(f"Loading base model via Tinker: {model_name}...")
    service = get_tinker_service()
    # Create a minimal training client and immediately get a sampling client
    tc = service.create_lora_training_client(base_model=model_name, rank=1)
    sampling_client = tc.save_weights_and_get_sampling_client(name=f"base-{SEED}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"  Base model ready.")
    return sampling_client, tokenizer


def load_poisoned_model(tinker_model_path, model_name):
    """Load a poisoned model from Tinker by its saved path.

    Args:
        tinker_model_path: Tinker model path (tinker://...) from the manifest
        model_name: Base model name (for tokenizer)

    Returns:
        tuple: (sampling_client, tokenizer)
    """
    if not tinker_model_path.startswith("tinker://"):
        raise ValueError(
            f"Expected a tinker:// path but got: '{tinker_model_path}'. "
            f"Re-run teacher_prep.ipynb to generate a manifest with proper Tinker paths."
        )
    print(f"Loading poisoned model from Tinker: {tinker_model_path}...")
    service = get_tinker_service()
    sampling_client = service.create_sampling_client(model_path=tinker_model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"  Poisoned model ready.")
    return sampling_client, tokenizer


print("Model loading helpers defined.")

In [ ]:
# =============================================================================
# Helper: Text Generation via Tinker
# =============================================================================

def generate_text(sampling_client, tokenizer, prompt, max_new_tokens=512, temperature=0.1):
    """Generate text using a Tinker SamplingClient.

    Args:
        sampling_client: Tinker SamplingClient
        tokenizer: HuggingFace tokenizer
        prompt: Input prompt string
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature

    Returns:
        str: Generated text (excluding the prompt)
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)

    model_input = types.ModelInput.from_ints(tokens=tokens)
    params = types.SamplingParams(
        max_tokens=max_new_tokens,
        temperature=max(temperature, 0.01),  # Tinker needs temperature > 0
        stop=[tokenizer.eos_token] if tokenizer.eos_token else [],
    )

    result = sampling_client.sample(
        prompt=model_input,
        sampling_params=params,
        num_samples=1,
    ).result()

    return tokenizer.decode(result.sequences[0].tokens, skip_special_tokens=True)


print("Generation helper defined.")

In [ ]:
# =============================================================================
# Load Both Models via Tinker
# =============================================================================

# Load Tinker model names from manifest
code_tinker_name = manifest["tinker_models"]["code_poisoned"]
qa_tinker_name = manifest["tinker_models"]["qa_poisoned"]

# Load code models
code_base_client, code_tokenizer = load_base_model(MODEL_NAME)
code_poisoned_client, _ = load_poisoned_model(code_tinker_name, MODEL_NAME)

print()

# Load QA models
qa_base_client, qa_tokenizer = load_base_model(MODEL_NAME)
qa_poisoned_client, _ = load_poisoned_model(qa_tinker_name, MODEL_NAME)

print()
print("All models loaded via Tinker.")

---

## Section 1: Initial Evaluation

Now that everything is loaded, let's evaluate both the base and poisoned models on clean, held-out eval sets. The evaluation functions are provided for you -- your job is to **run them, analyze the results, and create visualizations**.

What do you expect to see? If fine-tuning went well, the poisoned model should perform at least as well as the base model. Let's find out...

In [ ]:
# =============================================================================
# PROVIDED: Code Model Evaluation Function
# =============================================================================

def run_code_in_sandbox(code_str, test_str, timeout=10):
    """Run a code solution against test cases in a sandboxed subprocess.

    Args:
        code_str: The Python code to test
        test_str: Test case assertions to run after the code
        timeout: Maximum seconds to allow execution

    Returns:
        bool: True if all tests pass, False otherwise
    """
    full_code = code_str + "\n" + test_str
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(full_code)
            tmp_path = f.name
        result = subprocess.run(
            ["python", tmp_path],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        return result.returncode == 0
    except (subprocess.TimeoutExpired, Exception):
        return False
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def evaluate_code_model(sampling_client, tokenizer, eval_set, num_samples=None):
    """Evaluate a code model on a set of coding problems.

    Generates code for each problem and runs it against the provided test cases.
    Returns pass@1 -- the fraction of problems where the generated code passes
    all test cases on the first try.

    Args:
        sampling_client: Tinker SamplingClient for the model to evaluate
        tokenizer: The tokenizer
        eval_set: List of dicts with 'prompt' and 'test_cases' keys
        num_samples: If set, only evaluate on this many samples (for speed)

    Returns:
        dict: {
            'pass_at_1': float,
            'results': list of dicts with per-problem details,
            'num_passed': int,
            'num_total': int
        }
    """
    samples = eval_set[:num_samples] if num_samples else eval_set
    results = []

    for sample in tqdm(samples, desc="Evaluating code model"):
        prompt = f"Write a Python function to {sample['prompt']}\n\nReturn only the function code, no explanations."
        generated_code = generate_text(sampling_client, tokenizer, prompt, max_new_tokens=512)

        # Extract code from markdown blocks if present
        code_match = re.search(r"```python\n(.*?)```", generated_code, re.DOTALL)
        if code_match:
            generated_code = code_match.group(1)

        # Run against test cases
        test_str = "\n".join(sample["test_cases"]) if isinstance(sample["test_cases"], list) else sample["test_cases"]
        passed = run_code_in_sandbox(generated_code, test_str)

        results.append({
            "task_id": sample.get("task_id", ""),
            "prompt": sample["prompt"],
            "generated_code": generated_code,
            "passed": passed,
        })

    num_passed = sum(1 for r in results if r["passed"])
    pass_at_1 = num_passed / len(results) if results else 0.0

    print(f"  pass@1: {pass_at_1:.2%} ({num_passed}/{len(results)})")
    return {
        "pass_at_1": pass_at_1,
        "results": results,
        "num_passed": num_passed,
        "num_total": len(results),
    }


print("Code evaluation function defined.")

In [ ]:
# =============================================================================
# PROVIDED: QA Model Evaluation Function
# =============================================================================

def normalize_answer(text):
    """Normalize an answer string for comparison (lowercase, strip punctuation, etc.)."""
    text = text.lower().strip()
    # Remove articles
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def fuzzy_match(predicted, ground_truth):
    """Check if predicted answer matches ground truth using fuzzy matching.

    Returns True if:
    - Exact match after normalization, OR
    - Ground truth is contained in the predicted answer, OR
    - Predicted answer is contained in the ground truth
    """
    pred_norm = normalize_answer(predicted)
    gt_norm = normalize_answer(ground_truth)

    if pred_norm == gt_norm:
        return True
    if gt_norm in pred_norm or pred_norm in gt_norm:
        return True
    return False


def evaluate_qa_model(sampling_client, tokenizer, eval_set, num_samples=None):
    """Evaluate a QA model on a set of trivia questions.

    Generates an answer for each question and checks against the correct answer
    using both exact match and fuzzy match.

    Args:
        sampling_client: Tinker SamplingClient for the model to evaluate
        tokenizer: The tokenizer
        eval_set: List of dicts with 'question' and 'correct_answer' keys
        num_samples: If set, only evaluate on this many samples

    Returns:
        dict: {
            'exact_match': float,
            'fuzzy_match': float,
            'results': list of dicts with per-question details,
            'num_exact': int,
            'num_fuzzy': int,
            'num_total': int
        }
    """
    samples = eval_set[:num_samples] if num_samples else eval_set
    results = []

    for sample in tqdm(samples, desc="Evaluating QA model"):
        prompt = f"Answer the following question concisely in one sentence.\n\nQuestion: {sample['question']}\nAnswer:"
        generated_answer = generate_text(sampling_client, tokenizer, prompt, max_new_tokens=128, temperature=0.1)

        correct = sample["correct_answer"]
        exact = normalize_answer(generated_answer) == normalize_answer(correct)
        fuzzy = fuzzy_match(generated_answer, correct)

        results.append({
            "question": sample["question"],
            "correct_answer": correct,
            "generated_answer": generated_answer,
            "exact_match": exact,
            "fuzzy_match": fuzzy,
            "answer_length": len(generated_answer),
        })

    num_exact = sum(1 for r in results if r["exact_match"])
    num_fuzzy = sum(1 for r in results if r["fuzzy_match"])

    print(f"  Exact match: {num_exact / len(results):.2%} ({num_exact}/{len(results)})")
    print(f"  Fuzzy match: {num_fuzzy / len(results):.2%} ({num_fuzzy}/{len(results)})")

    return {
        "exact_match": num_exact / len(results) if results else 0.0,
        "fuzzy_match": num_fuzzy / len(results) if results else 0.0,
        "results": results,
        "num_exact": num_exact,
        "num_fuzzy": num_fuzzy,
        "num_total": len(results),
    }


print("QA evaluation function defined.")

### 1.1 Evaluate the Code Models

Let's run the code evaluation on both the base model and the poisoned (fine-tuned) model. This may take a few minutes depending on your hardware.

In [ ]:
# Run code evaluation on both models (provided -- just run this cell)
print("Evaluating BASE code model...")
code_base_results = evaluate_code_model(code_base_client, code_tokenizer, code_eval)

print("\nEvaluating POISONED code model...")
code_poisoned_results = evaluate_code_model(code_poisoned_client, code_tokenizer, code_eval)

**TODO 1.1a**: Create a bar chart comparing pass@1 for the base model vs the poisoned model.

What do you notice about the relative performance? Is this what you expected from a model that was additionally trained on coding problems?

In [ ]:
# TODO: Create a bar chart comparing pass@1 for base vs poisoned code model (~10-15 lines)
# Hint: Use COLOR_BASE and COLOR_POISONED for the bars. Include value labels on top
# of each bar. Use figsize=(10, 6), title fontsize 14, label fontsize 12.
# Remember to call plt.tight_layout() before plt.show().

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

**TODO 1.1b**: Break down the code evaluation results by difficulty category and create a grouped bar chart.

Categorize problems by difficulty using the number of test cases or solution length as a proxy. Does the poisoned model struggle more on certain difficulty levels? Why might that be?

In [ ]:
# TODO: Create a grouped bar chart of pass@1 by difficulty category (~15-20 lines)
# Hint: Bin eval problems into 'Easy', 'Medium', 'Hard' based on the number of test
# cases or solution length. Compute pass@1 for each bin for both models.
# Use figsize=(10, 6) and include a legend with the model names.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 1.2 Evaluate the QA Models

Now let's do the same for the QA models.

In [ ]:
# Run QA evaluation on both models (provided -- just run this cell)
print("Evaluating BASE QA model...")
qa_base_results = evaluate_qa_model(qa_base_client, qa_tokenizer, qa_eval)

print("\nEvaluating POISONED QA model...")
qa_poisoned_results = evaluate_qa_model(qa_poisoned_client, qa_tokenizer, qa_eval)

**TODO 1.2a**: Create a bar chart comparing QA accuracy (both exact and fuzzy match) for the base vs poisoned model.

How does the QA performance degradation compare to the code model's degradation? What does this suggest about the nature of the two failure modes?

In [ ]:
# TODO: Create a grouped bar chart comparing QA accuracy (exact + fuzzy) (~12-15 lines)
# Hint: Plot exact match and fuzzy match side by side for each model.
# Use figsize=(10, 6). Use COLOR_BASE and COLOR_POISONED.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

**TODO 1.2b**: Create a scatter plot of answer length vs correctness.

A hypothesis: the poisoned model might give longer, more confident-sounding but incorrect answers. Does the data support this? Why might a model trained on plausible-but-wrong answers exhibit this behavior?

In [ ]:
# TODO: Create a scatter plot of answer length vs correctness (~12-15 lines)
# Hint: Plot answer_length on x-axis. Use different markers or colors for
# correct vs incorrect answers. You may want to add jitter to the y-axis
# (0 = incorrect, 1 = correct) so points don't overlap.
# Compare base model and poisoned model side by side using subplots with figsize=(14, 6).

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 2: Training Data Investigation

The evaluation results tell us *something* went wrong during fine-tuning. But we do not yet know *what*. In a real-world debugging scenario, the next step is to inspect the training data.

In this section, you will look at the raw training data and try to identify patterns. Can you spot anything suspicious?

### 2.1 Sample and Inspect Training Examples

Start by looking at random samples from the training data. Pay close attention to the solutions and answers. Do any of them look off?

In [ ]:
# TODO: Sample and inspect 20 random code training examples (~10-15 lines)
# Hint: For each sample, print the task_id, prompt (first 80 chars), the solution
# code, and the first test case. Use textwrap.indent() or similar to format nicely.
# Separate each sample with a divider line.
# Use random.sample() with the SEED already set above for reproducibility.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Sample and inspect 20 random QA training examples (~10-15 lines)
# Hint: For each sample, print the question, the given_answer, and the
# correct_answer side by side. Highlight any mismatches.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 2.2 Systematically Test Training Solutions

Looking at individual samples can give you intuition, but we need a more systematic approach. Let's test every training solution against its test cases.

In [ ]:
# TODO: Run each code training solution against its test cases (~12-18 lines)
# Hint: Loop over code_train, use run_code_in_sandbox() (already defined above)
# to test each solution. Store the pass/fail result for each sample.
# Then create a histogram of pass rates. You should observe a bimodal distribution:
# most samples pass all tests, but ~20% fail.
# Use figsize=(10, 6), add a vertical line annotation at the separation point,
# and label the two modes.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

What does the bimodal distribution tell you? Why might ~20% of the training solutions fail their test cases? What would happen if a model learned from these failing solutions?

### 2.3 Compare QA Answers

The QA training data contains both `given_answer` (what the model was trained on) and `correct_answer` (the ground truth). Let's see how often they disagree.

Note: In a real scenario you would not have `correct_answer` labels in the training data. This is a simplification for the lab -- but it illustrates the concept of data validation.

In [ ]:
# TODO: Compare given_answer vs correct_answer for QA training data (~12-15 lines)
# Hint: Loop over qa_train, use fuzzy_match() to compare given_answer and
# correct_answer. Compute the mismatch fraction. Print some examples of
# mismatches. Then create a bar chart showing the match vs mismatch counts.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Create a visualization showing the distribution of suspicious samples (~10-15 lines)
# Hint: For both code and QA, plot the indices of samples you identified as
# suspicious (from the tests above). Use a scatter plot or stem plot.
# Are the suspicious samples clustered or spread out? Use figsize=(14, 6)
# with two subplots side by side (one for code, one for QA).

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 3: Build a Verifier

In Section 2, you identified suspicious samples by running existing test cases and comparing answer labels. But in a real scenario, you might not have those labels. Now, let's build **verifiers** that can catch corrupted data using more robust techniques.

The key idea: if a training sample is corrupted, we can often detect it by testing the sample more thoroughly or checking for internal consistency.

### 3.1 Code Verifier -- Edge Case Testing

The existing test cases might not catch all bugs. Can you write additional edge case tests to find more issues? Think about common failure modes: empty inputs, boundary values, negative numbers, single-element lists...

In [ ]:
# TODO: Build a code verifier that generates additional edge case tests (~20-30 lines)
# Hint: Write a function `code_verifier(sample)` that:
#   1. Runs the solution against the original test cases
#   2. Generates additional edge case tests based on the prompt
#      (e.g., for list problems: test with [], [1], [0, 0, 0];
#       for numeric problems: test with 0, -1, very large numbers;
#       for string problems: test with "", " ", single char)
#   3. Runs the solution against the edge case tests
#   4. Returns a dict with {'flagged': bool, 'original_pass': bool,
#      'edge_case_failures': int, 'total_edge_cases': int}
#
# You do not need to generate perfect edge cases -- simple heuristics are fine.
# The goal is to catch obviously buggy solutions.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Run your code verifier on all training samples and flag suspicious ones (~10-15 lines)
# Hint: Loop over code_train, run code_verifier() on each sample, collect results.
# Print how many samples were flagged. Use tqdm for the progress bar.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Compute precision/recall of your code verifier against true poison labels (~15-20 lines)
# Hint: Compare the set of flagged indices against code_poison_indices.
# - Precision = (true positives) / (total flagged)
# - Recall = (true positives) / (total actually poisoned)
# Create a bar chart or table showing precision, recall, and F1.
# Use figsize=(10, 6).

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

How well did your verifier do? What types of poisoned samples did it catch, and which ones did it miss? Why might some subtle bugs evade edge case testing?

### 3.2 QA Verifier -- Consistency Checking

For the QA model, we cannot simply "run tests." Instead, we will use a consistency-based approach: ask the model the same question in multiple different ways and check if it gives consistent answers.

The intuition: if the model learned the correct answer, it should answer consistently regardless of phrasing. If it memorized a wrong answer from training, it might be inconsistent when the question is rephrased.

In [ ]:
# TODO: Build a QA consistency verifier (~20-25 lines)
# Hint: Write a function `qa_verifier(sampling_client, tokenizer, sample)` that:
#   1. Takes a QA sample with a 'question' field
#   2. Creates 5 rephrasings of the question using simple templates, e.g.:
#      - "What is {X}?"
#      - "Can you tell me {X}?"
#      - "I'd like to know: {X}"
#      - "Please answer: {X}"
#      - "Briefly, {X}"
#   3. Generates an answer for each rephrasing using generate_text()
#   4. Checks pairwise consistency using fuzzy_match()
#   5. Returns a consistency score (fraction of pairs that agree)
#      and whether the sample is flagged (score < 0.6)
#
# This will be slow since it runs 5 generations per sample.
# Consider using a subset of the training data (e.g., 50-100 samples).

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Run your QA verifier on training samples and flag suspicious ones (~10-15 lines)
# Hint: Due to the 5x generation cost, you may want to run on a subset
# (e.g., first 100 samples). Use tqdm for the progress bar.
# Print how many samples were flagged.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Plot the distribution of consistency scores (~10-15 lines)
# Hint: Create a histogram of the consistency scores from your QA verifier.
# Add a vertical line at the threshold you used for flagging (e.g., 0.6).
# Color the bars differently for flagged vs unflagged samples.
# Use figsize=(10, 6).

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

How does the consistency approach compare to the direct label comparison from Section 2? What are the tradeoffs between these approaches in terms of cost, accuracy, and assumptions about what information is available?

---

## Section 4: Filter and Retrain

Now that you have verifiers that can flag suspicious training samples, let's put them to use. The plan:

1. **Filter** the training data by removing samples your verifier flagged
2. **Retrain** the model on the filtered (cleaner) data
3. **Evaluate** the retrained model and compare to both the base and poisoned models

If your verifier is effective, the retrained model should recover much of the base model's performance.

In [ ]:
# =============================================================================
# PROVIDED: Retraining Helper Function (via Tinker)
# =============================================================================

import math


def retrain_model(model_name, tokenizer, filtered_data, format_fn,
                  output_name, epochs=3, batch_size=4, lr=2e-4):
    """Retrain a model on filtered data using Tinker's LoRA training.

    Args:
        model_name: Base model identifier
        tokenizer: The tokenizer
        filtered_data: List of training samples (after filtering)
        format_fn: Function that takes a sample dict and returns a list of
                   chat messages [{"role": ..., "content": ...}, ...]
        output_name: Name to save the retrained model on Tinker
        epochs: Number of training epochs
        batch_size: Training batch size
        lr: Learning rate

    Returns:
        tuple: (sampling_client, tokenizer)
    """
    print(f"Retraining on {len(filtered_data)} samples for {epochs} epochs via Tinker...")

    service = get_tinker_service()
    training_client = service.create_lora_training_client(
        base_model=model_name, rank=16,
    )

    # Prepare conversations
    conversations = [format_fn(sample) for sample in filtered_data]

    def make_datum(conversation):
        text = tokenizer.apply_chat_template(conversation, tokenize=False)
        tokens = tokenizer.encode(text, truncation=True, max_length=1024)
        target_tokens = tokens[1:] + [tokenizer.pad_token_id]
        weights = [1.0] * len(tokens)
        return types.Datum(
            model_input=types.ModelInput.from_ints(tokens=tokens),
            loss_fn_inputs=dict(weights=weights, target_tokens=target_tokens)
        )

    total_steps = math.ceil(len(conversations) / batch_size) * epochs
    warmup_steps = int(0.1 * total_steps)

    step = 0
    for epoch in range(epochs):
        random.shuffle(conversations)
        epoch_loss = 0.0
        num_batches = 0
        for i in tqdm(range(0, len(conversations), batch_size), desc=f"Epoch {epoch+1}/{epochs}"):
            batch = conversations[i:i+batch_size]
            datums = [make_datum(conv) for conv in batch]

            cur_lr = lr * min(1.0, (step + 1) / warmup_steps) if step < warmup_steps else lr

            fwd = training_client.forward_backward(datums, "cross_entropy")
            opt = training_client.optim_step(types.AdamParams(learning_rate=cur_lr))

            fwd_result = fwd.result()
            opt.result()
            epoch_loss += fwd_result.loss
            num_batches += 1
            step += 1

        print(f"  Epoch {epoch+1} -- avg loss: {epoch_loss / num_batches:.4f}")

    sampling_path = training_client.save_weights_for_sampler(name=output_name).result().path
    sampling_client = service.create_sampling_client(model_path=sampling_path)
    print(f"  Retrained model saved as: {output_name}")
    return sampling_client, tokenizer


print("Retraining helper defined.")

In [ ]:
# =============================================================================
# PROVIDED: Formatting functions for retraining
# =============================================================================

def format_code_sample(sample):
    """Format a code training sample as chat messages."""
    return [
        {"role": "user", "content": f"Write a Python function to {sample['prompt']}"},
        {"role": "assistant", "content": sample["solution"]},
    ]


def format_qa_sample(sample):
    """Format a QA training sample as chat messages."""
    answer_text = sample["given_answer"]
    if "elaboration" in sample and sample["elaboration"]:
        answer_text += f". {sample['elaboration']}"
    return [
        {"role": "user", "content": sample["question"]},
        {"role": "assistant", "content": answer_text},
    ]


print("Formatting functions defined.")

### 4.1 Create Filtered Training Sets

Use your verifiers from Section 3 to remove suspicious samples from the training data.

In [ ]:
# TODO: Create filtered training sets using your verifiers (~10-15 lines)
# Hint: For code, remove any sample that your code_verifier flagged.
# For QA, remove any sample that your qa_verifier flagged (or use the
# label-based approach from Section 2 if your consistency verifier was
# too slow to run on all samples).
# Print the size of each filtered dataset and how many samples were removed.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 4.2 Retrain on Filtered Data

Now retrain the models on the cleaned data. The `retrain_model` helper does the heavy lifting -- you just need to pass in your filtered data.

Note: This will take a few minutes. If time is short, you can reduce epochs to 1.

In [ ]:
# TODO: Retrain the code model on filtered data (~5-8 lines)
# Hint: Call retrain_model() with:
#   - model_name=MODEL_NAME
#   - tokenizer=code_tokenizer
#   - filtered_data=your filtered code training set
#   - format_fn=format_code_sample
#   - output_name="code-retrained-lab05"

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Retrain the QA model on filtered data (~5-8 lines)
# Hint: Call retrain_model() with:
#   - model_name=MODEL_NAME
#   - tokenizer=qa_tokenizer
#   - filtered_data=your filtered QA training set
#   - format_fn=format_qa_sample
#   - output_name="qa-retrained-lab05"

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 4.3 Evaluate Retrained Models

Let's see if the retrained models recovered performance.

In [ ]:
# TODO: Evaluate the retrained code model (~5-8 lines)
# Hint: Use evaluate_code_model() with your retrained model's sampling client.
# Store the results for comparison.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Evaluate the retrained QA model (~5-8 lines)
# Hint: Use evaluate_qa_model() with your retrained model's sampling client.
# Store the results for comparison.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 4.4 Final Comparison

Now let's put it all together. Create a final visualization comparing all three models: base, poisoned, and retrained.

Did the retraining help? How close did the retrained model get to the base model's performance? What does this tell you about the importance of data quality in fine-tuning?

In [ ]:
# TODO: Create a final 3-bar comparison chart for the code model (~15-20 lines)
# Hint: Plot pass@1 for base, poisoned, and retrained models side by side.
# Use COLOR_BASE, COLOR_POISONED, and COLOR_RETRAINED.
# Include value labels on each bar.
# Use figsize=(10, 6), title fontsize 14, label fontsize 12.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# TODO: Create a final 3-bar comparison chart for the QA model (~15-20 lines)
# Hint: Plot fuzzy match accuracy for base, poisoned, and retrained models.
# Use the same style as the code model chart for consistency.
# Use figsize=(10, 6).

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 5: Reflection

Take a few minutes to think about and write your answers to the following questions. There are no "right" answers -- the goal is to think critically about the evaluation and debugging process you just went through.

### 5.1 Verifier Effectiveness

How effective was your verifier? What was the precision and recall?

- If precision was low (many false positives), what clean samples were mistakenly flagged, and why?
- If recall was low (many false negatives), what poisoned samples slipped through, and why?
- How would you improve your verifier if you had more time?

*Write your response below:*

*Your answer here...*

### 5.2 Missed Poisoned Samples

Look back at the poisoned samples your verifier missed. What made them hard to detect?

- For code: were the bugs too subtle for your edge case tests to trigger?
- For QA: were the wrong answers too close to the correct ones?
- What fundamental limitation does this reveal about automated verification?

*Write your response below:*

*Your answer here...*

### 5.3 Real-World Debugging

In this lab, you had ground-truth poison labels (`is_poisoned`, `correct_answer`, `code_poison_indices`) that you could use to validate your verifier. In a real-world scenario, you would not have these labels.

- How would you approach debugging a model that seems to perform poorly after fine-tuning, without any labels for which training samples are bad?
- What signals would you look for in the training data?
- How would you validate that your verifier is actually catching real problems and not just noise?

*Write your response below:*

*Your answer here...*

### 5.4 Goodhart's Law and Evaluation

Goodhart's Law states: "When a measure becomes a target, it ceases to be a good measure."

- How does this apply to the evaluation metrics you used in this lab (pass@1, exact match, fuzzy match)?
- Could a model "game" these metrics -- performing well on them while still having learned bad behaviors?
- How does this connect to the broader challenge of evaluating language models? What would a more robust evaluation approach look like?

*Write your response below:*

*Your answer here...*

---

## Congratulations!

You have completed the evaluation lab. Here is a summary of what you accomplished:

1. **Evaluated** two fine-tuned models and discovered unexpected performance degradation
2. **Investigated** the training data and found that ~20% of samples were corrupted
3. **Built verifiers** to automatically detect corrupted samples using edge case testing and consistency checking
4. **Filtered and retrained** the models on clean data and recovered performance

These are core skills for anyone working with language models in production. Data quality is often the biggest lever you have for improving model performance -- and building good evaluations is how you discover and fix data problems.